In [1]:
from GlobalCensus.data.USA.census import USCensus
import GlobalCensus.data.USA.processing as processing
import GlobalCensus.core.geometry_utils as geometry_utils

In [2]:
api_key = "7a5f64a4565e83e9742bebb89474aa22a349e91d" # Get yours here https://api.census.gov/data/key_signup.html

In [3]:
cache_path = "us_census_cache" 
pygris_path = "us_census_cache/pygris" 

city_name = "Boston, MA, USA"

In [4]:
aoi = geometry_utils.get_city_geometry(city_name)

### Layer Types in `USCensus`

**Spinal layers**
These are the core census hierarchy levels where official datasets are natively available (e.g. county, tract, block group). They form the “backbone” of the system, and all primary census attributes are directly retrieved at these resolutions.

**Named (derived) layers**
These are additional geometries that do not have native census data. Instead, data is **resampled** from a compatible spinal layer (typically the finest available, such as block group). This allows arbitrary geographies (e.g. custom zones, service areas) to inherit census attributes through spatial aggregation.

**Dataset availability nuance**
Not all datasets are available at every spinal level. For example, some datasets (like ACM) may exist for tract or block group, but **not for block level**. In such cases, the system relies on the nearest valid spinal layer as the source for resampling.


In [5]:
census = USCensus(
    aoi=aoi,
    levels=["county","tract"], # Census layer
    pygris_cache_dir=pygris_path, # Cache for geometry files in pygris
    census_cache_dir=cache_path, # Cache for census parquet files
    api_key=api_key, # US Census API Key. Get yours here https://api.census.gov/data/key_signup.html
    erase_water=[False,True] # You can choose if you want to erase water form geometries or not.
    # Here do not erase water for county and do it for tract
)

Using FIPS code '25' for input 'Massachusetts'
USCensus: Fetching county...


/home/miguel/Documents/Proyectos/PTLevelofService/GlobalCensus/src/GlobalCensus/data/USA/processing.py:144: UserWarning: Extension type 'geoarrow.wkb' is not registered; loading as its storage type.

To avoid this warning, register the extension type or set environment variable 'POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR' to 'load_as_storage' or 'load_as_extension'.

In Polars 2.0, the default behavior will change to 'load_as_extension'.
  return lf.with_columns(agg_exprs).select(["GEOID", "geometry", "year", "state", "area"] + requested_logical_names).collect()


USCensus: Fetching tract...


Add new census layer

In [6]:
census.add_level("state")

USCensus: Fetching state...


Add new census level that is not spinal

As we did not donwload blockgroup (the layer place resamples from) this layer will be downloaded too.

In [7]:
census.add_level("place")

USCensus: Fetching blockgroup...
Using FIPS code '25' for input 'MA'
Using FIPS code '25' for input 'MA'


/home/miguel/Documents/Proyectos/PTLevelofService/GlobalCensus/.venv/lib/python3.12/site-packages/geopandas/tools/overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 62 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)


USCensus: Fetching place...
Using FIPS code '25' for input 'Massachusetts'
Using FIPS code '25' for input 'Massachusetts'
Using FIPS code '25' for input 'Massachusetts'
Using FIPS code '25' for input 'Massachusetts'


Add an arbitrary GeoDataFrame as census layer

In [8]:
census.add_layer(aoi,name="area_of_interest",agg_from="blockgroup")